In [11]:
%pip install anndata

In [12]:
import numpy as np
from tqdm import tqdm
import pandas as pd
import anndata
import csv
import sys
import logging
import os
from matplotlib import pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from scipy.stats import pearsonr

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
# path to the DE files produced by the DE notebook

stim_de_file = "/path/to/stim_DE_D1_v3.txt"
stim_df1 = pd.read_csv(stim_de_file, sep='\t')

genotype_de_file =  "/path/to/PTC_DE_D1_v3.txt"
geno_df1 = pd.read_csv(genotype_de_file , sep='\t')

stim_de_file = "/path/to/stim_DE_D1_v3.txt"
stim_df2 = pd.read_csv(stim_de_file, sep='\t')

genotype_de_file =  "/path/to/PTC_DE_D1_v3.txt"
geno_df2 = pd.read_csv(genotype_de_file , sep='\t')

In [15]:
variants = set(col.split('_')[0]+ '_' + col.split('_')[1] +  '_' + col.split('_')[2] for col in geno_df1.columns if '_logfc' in col or '_pval' in col)

filtered_variants = [
    s for s in variants if (s.endswith("_het") or s.endswith("_hom")) and not s.startswith("Ctrl_")
]

In [16]:
variants_to_plot = filtered_variants

sig_level = 1e-30
logfc_level = 0.5

correlation_df = pd.DataFrame(index=variants_to_plot, columns=variants_to_plot)

for variant1 in tqdm(variants_to_plot):
  for variant2 in variants_to_plot:

    stim1 = variant1.split('_')[0]
    genes1_pval = stim_df1['gene_names'][stim_df1[stim1 + '_pval_adj'] < sig_level].values
    genes1_logfc = stim_df1['gene_names'][stim_df1[stim1 + '_logfc'] > logfc_level].values
    genes1 = np.intersect1d(genes1_pval, genes1_logfc)

    stim2 = variant2.split('_')[0]
    genes2_pval = stim_df2['gene_names'][stim_df2[stim2 + '_pval_adj'] < sig_level].values
    genes2_logfc = stim_df2['gene_names'][stim_df2[stim2 + '_logfc'] > logfc_level].values
    genes2 = np.intersect1d(genes2_pval, genes2_logfc)

    genes1 = geno_df1['gene_name']
    genes2 = geno_df2['gene_name']

    union_genes = np.union1d(genes1, genes2)

    df1_filtered = geno_df1[geno_df1['gene_name'].isin(union_genes)]
    df2_filtered = geno_df2[geno_df2['gene_name'].isin(union_genes)]

    # Sort the dataframes by gene name to align them
    df1_filtered = df1_filtered.set_index('gene_name').loc[union_genes].reset_index()
    df2_filtered = df2_filtered.set_index('gene_name').loc[union_genes].reset_index()

    # Extract the logFC values
    logFC_values1 = df1_filtered[variant1 + '_logfc']
    logFC_values2 = df2_filtered[variant2 + '_logfc']

    # Compute Pearson's correlation
    if not logFC_values1.empty and not logFC_values2.empty:  # Ensure there are values to compare
      try:
        correlation, _ = pearsonr(logFC_values1, logFC_values2)
      except:
        correlation = 0
    else:
      correlation = 0  # Set as NaN if there's no data to correlate

    # Assign the correlation to the DataFrame
    correlation_df.loc[variant1, variant2] = correlation

100%|██████████| 180/180 [12:10<00:00,  4.06s/it]


In [17]:
correlation_df

,TGFb_chr9:117708561:A/G_hom,Pam3CSK4_chr2:102166111:A/G_hom,IL10_chr3:38139029:G/A_hom,TGFb_chr9:117712387:A/G_hom,LPS_chr4:153702934:G/A_het,Pam3CSK4_chr9:117712387:A/G_het,TGFb_chr4:153702908:A/G_hom,IL10_chr5:132487088:C/T_hom,IFNG_chr3:38139029:G/A_het,IL1b_chr9:99132634:C/T_hom,...,IL10_chr2:102166111:A/G_hom,IL1b_chr9:99132634:C/T_het,LPS_chr11:117993239:A/G_het,Pam3CSK4_chr3:38139029:G/A_het,IFNG_chr9:117708561:A/G_het,LPS_chr3:38140849:G/A_hom,TGFb_chr11:117986535:G/A_hom,IL10_chr9:99132634:C/T_hom,TGFb_chr11:117993239:A/G_hom,LPS_chr9:117708561:A/G_het
TGFb_chr9:117708561:A/G_hom,-0.115315,-0.001195,-0.132846,-0.284734,-0.02664,-0.022751,-0.219106,-0.017006,-0.070413,0.032003,...,-0.109487,-0.048116,0.029409,-0.037722,0.015981,-0.056854,-0.016905,0.049954,-0.105396,0.057174
Pam3CSK4_chr2:102166111:A/G_hom,0.025307,0.069929,0.023674,0.027375,0.063075,-0.035195,-0.029118,-0.045523,-0.02868,-0.002445,...,0.007888,0.0167,-0.038682,-0.157126,-0.070024,0.02834,0.018212,-0.017263,-0.045811,0.059903
IL10_chr3:38139029:G/A_hom,-0.001958,-0.0127,0.193796,0.143519,0.022233,-0.039489,0.029834,-0.060544,0.141307,0.053381,...,0.01462,0.098604,0.006696,-0.007775,0.062318,0.091542,0.067391,-0.143151,0.047846,-0.009284
TGFb_chr9:117712387:A/G_hom,0.049315,0.12449,0.070936,0.034785,0.109257,-0.006491,-0.058705,0.047517,0.128461,-0.046482,...,0.064711,-0.221055,-0.226261,-0.013349,0.088637,-0.0661,0.043507,0.04852,0.010912,0.004258
LPS_chr4:153702934:G/A_het,-0.026355,-0.187082,0.012279,0.043132,0.021971,-0.029693,0.105168,0.182737,-0.060441,0.033335,...,-0.052341,-0.025891,-0.000965,-0.019605,-0.043812,0.047013,0.033399,0.018226,0.139885,-0.014561
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
LPS_chr3:38140849:G/A_hom,-0.022827,-0.116813,0.016396,-0.023889,-0.000469,-0.073935,0.024957,0.138104,-0.044585,0.145695,...,-0.008321,-0.034362,0.086786,0.020522,-0.042458,0.178004,0.122897,0.020379,0.004909,0.125211
TGFb_chr11:117986535:G/A_hom,0.04333,-0.047259,-0.014139,0.0363,0.010158,0.074782,0.032054,0.039448,0.116789,-0.070583,...,-0.014501,-0.177088,0.041329,0.040618,0.086021,0.036714,0.102066,-0.089963,0.159848,-0.042074
IL10_chr9:99132634:C/T_hom,-0.058308,0.124261,-0.123106,-0.034669,0.010973,0.030256,0.017155,-0.085,-0.017206,0.226325,...,-0.111674,-0.025865,0.114706,0.054275,0.039794,-0.003903,0.014342,-0.043768,0.058945,0.070471
TGFb_chr11:117993239:A/G_hom,-0.008907,-0.080192,0.051147,0.004417,0.031884,-0.034729,0.009766,0.077692,0.043319,0.148533,...,-0.007595,-0.160032,0.004324,-0.022773,-0.01285,0.070207,0.280965,-0.030913,0.048159,0.091584


In [18]:
correlation_df.to_csv('correlation_table.csv')